# Exploring sample trip logs & testing PySpark logic

In [1]:
import requests
import zipfile
import io
import os
import pandas as pd
from pyspark.sql import SparkSession

In [2]:
# Define local mock Azure Data Lake path
bronze_historical_path = "../data/bronze/historical/"
os.makedirs(bronze_historical_path, exist_ok=True)

# Define the two sample months (1 winter, 1 summer) from the AWS bucket
zip_urls = [
    "https://s3.amazonaws.com/tripdata/202401-citibike-tripdata.zip",
    "https://s3.amazonaws.com/tripdata/202407-citibike-tripdata.zip"
]

In [7]:
print("Downloading and extracting ZIP files...")
for url in zip_urls:
    print(f"Fetching {url}...")
    response = requests.get(url)

    # Extract all CSVs (handling months with >1 million trips)
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        z.extractall(bronze_historical_path)

print(f"Extraction complete. Files saved to {bronze_historical_path}")

Fetching https://s3.amazonaws.com/tripdata/202401-citibike-tripdata.zip...
Fetching https://s3.amazonaws.com/tripdata/202407-citibike-tripdata.zip...
Extraction complete. Files saved to ../data/bronze/historical/


In [3]:
# Initialize local Spark session mimicking Databricks

spark = SparkSession.builder \
    .appName("CitiBike-Batch-EDA") \
    .getOrCreate()

In [4]:
# Read all extracted CSVs in the historical folder
# PySpark will automatically merge split CSVs from the same month
df_trips = spark.read.option("header", "true") \
    .csv(f"{bronze_historical_path}/*.csv")

# Profile the schema to map out data types
df_trips.printSchema()

root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: string (nullable = true)
 |-- ended_at: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: string (nullable = true)
 |-- start_lng: string (nullable = true)
 |-- end_lat: string (nullable = true)
 |-- end_lng: string (nullable = true)
 |-- member_casual: string (nullable = true)



In [5]:
df_trips.count()

6610981

In [5]:
df_trips.limit(20).toPandas()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,8E865410DBDE0CA9,electric_bike,2024-01-01 13:00:04.563,2024-01-01 13:04:04.652,3 St & 3 Ave,4028.03,Carroll St & Smith St,4225.14,40.6750705,-73.98775226,40.680611,-73.99475825,casual
1,0403D0B3FC9CA77D,electric_bike,2024-01-08 19:36:43.520,2024-01-08 19:53:16.266,Franklin Ave & St Marks Ave,4107.05,Bedford Ave & Bergen St,4066.15,40.6758324,-73.9561677,40.676368,-73.952918,casual
2,F6DE7BB42FF550BE,electric_bike,2024-01-12 15:00:41.580,2024-01-12 15:36:29.622,W 67 St & Broadway,7116.04,Central Park W & W 103 St,7577.27,40.77492513,-73.98266566,40.79558954105342,-73.96188408136368,casual
3,84A995BFD98030D4,classic_bike,2024-01-12 16:52:19.025,2024-01-12 17:17:29.773,Central Park West & W 68 St,7079.06,E 5 St & Ave C,5545.04,40.7734066,-73.97782542,40.72299208,-73.97995466,member
4,7BBEAD4F2B535813,electric_bike,2024-01-05 19:50:19.202,2024-01-05 20:34:42.517,W 67 St & Broadway,7116.04,Ave A & E 14 St,5779.11,40.77492513,-73.98266566,40.73031071000895,-73.98047178983688,member
5,296A378A5C42028D,classic_bike,2024-01-03 17:45:12.017,2024-01-03 18:16:03.084,Central Park West & W 68 St,7079.06,E 5 St & Ave C,5545.04,40.7734066,-73.97782542,40.72299208,-73.97995466,member
6,9A891F9C924C240F,classic_bike,2024-01-05 09:47:22.093,2024-01-05 09:50:49.315,President St & Henry St,4307.13,Carroll St & Smith St,4225.14,40.6828003,-73.99990419,40.680611,-73.99475825,member
7,219B18A37682CC36,classic_bike,2024-01-06 08:35:40.778,2024-01-06 08:48:30.838,39 St & Queens Blvd,6218.04,44 Dr & 21 St,6319.03,40.74431,-73.92601,40.748036,-73.946705,member
8,9EA1D9C604956D7D,electric_bike,2024-01-12 07:59:57.689,2024-01-12 08:10:05.919,6 St & 7 Ave,3834.1,Lafayette Ave & Ft Greene Pl,4470.09,40.6686627,-73.97988067,40.68700231292619,-73.97664964199066,member
9,568F62EE667D0E39,classic_bike,2024-01-01 18:08:59.287,2024-01-01 18:13:09.837,W 43 St & 10 Ave,6756.01,9 Ave & W 39 St,6644.08,40.76009437,-73.99461843,40.756403523272496,-73.99410143494606,member
